In [1]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool

load_dotenv()


@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x**0.5


@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x**2


# create subagents
subagent_1 = create_agent(model="gpt-6-luna", tools=[square_root])

subagent_2 = create_agent(model="gpt-6-luna", tools=[square])


from langchain.messages import HumanMessage


@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square root of {x}")]}
    )
    return response["messages"][-1].content


@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square of {x}")]}
    )
    return response["messages"][-1].content


## Creating the main agent

main_agent = create_agent(
    model="gpt-6-luna",
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.",
)

In [2]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='e9213115-55fb-4c16-945a-732abdfe1a17'),
              AIMessage(content=[{'arguments': '{"x":456}', 'call_id': 'call_6kjYmZCOJQ33EDvgBmnzaQza', 'name': 'call_subagent_1', 'type': 'function_call', 'id': 'fc_048ba2a9515364cb006adbefeb0abc87d2bc30b0f09421e676', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_048ba2a9515364cb006adbefea337487d2b16f40c745086041', 'created_at': 1792798698.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_048ba2a9515364cb006adbefea337487d2b16f40c745086041', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_6kjYmZCOJQ33EDvgBmnzaQza', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 121, 'output_tokens': 21, 'total_tokens': 142, 'input_tok